# 🎓 Yonsei Colab Studio — 교육용 소형 LLM 파인튜닝

Google Colab **무료 T4 GPU**에서 [Unsloth](https://github.com/unslothai/unsloth)로 소형 LLM을 파인튜닝합니다.
메모리 사용량 ~80% 절감, 속도 2~3배로 무료 코랩에서도 OOM 없이 학습됩니다.

- **목적**: 20가지 교육 시나리오(소크라테스 문답 · 단계별 채점 · 학부모 안내문 변환 등) 특화
- **기본 모델**: `unsloth/Qwen2.5-1.5B-Instruct` (한국어 우수, 초경량)
- **방식**: LoRA(QLoRA) 4bit

> ⚠️ 먼저 상단 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 를 선택하세요.


## 0. GPU 확인


In [9]:
!nvidia-smi


Thu Jun 18 01:12:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             29W /   70W |    3911MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Unsloth 설치
코랩 최신 환경에 맞춰 자동 설치됩니다. (1~2분 소요)


In [10]:
%%capture
import os
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets


## 2. 학습 데이터 — HuggingFace Hub에서 받기
손으로 만든 예시가 아니라 **실제 한국어 교육 데이터셋**을 HF Hub에서 받아 20시나리오 포맷으로 변환합니다.

추천 데이터셋(소스별 N개만 streaming 추출 → 대형도 전체 다운로드 없음):
- `JosephLee/korean-socratic-qa` (105k) — 소크라테스 문답
- `kuotient/orca-math-word-problems-193k-korean` (193k) — 수학 단계 풀이
- `beomi/KoAlpaca-v1.1a` (21k) — 일반 instruction 베이스
- `jojo0217/korean_safe_conversation` (27k, Apache-2.0) — 정서지원·공감
- `neuralfoundry-coder/aihub-korean-education-instruct-sample` (6k) — 교육 상담·분석


In [11]:
# 저장소 clone (스크립트/레지스트리 사용)
REPO = 'https://github.com/xide-projext/xideproject.git'
import os
if not os.path.exists('xideproject'):
    !git clone -q $REPO
%cd xideproject

# HF에서 소스별 8000개씩 받아 data/hf_train.jsonl 생성 (≈5분)
!python scripts/fetch_hf_datasets.py --only code debug_en --per-source 1000 --val-ratio 0.05

from pathlib import Path
DATA_PATH = "data/hf_train.jsonl"
assert Path(DATA_PATH).exists(), "fetch 실패 — 위 셀 로그 확인"
print("학습 데이터 준비 완료:", DATA_PATH)


/content/xideproject/xideproject

▶ [code] m-a-p/CodeFeedback-Filtered-Instruction (목표 1000개, license=Apache-2.0)
  ✅ 1000개 수집

▶ [debug_en] taisazero/socratic-debugging-benchmark (목표 1000개, license=MIT)
  ✅ 815개 수집

총 1815개 수집 → 분할 저장
  저장: data/hf_train.jsonl (1725개)
  저장: data/hf_val.jsonl (90개)

[소스별 분포]
  m-a-p/CodeFeedback-Filtered-Instruction                 1000
  taisazero/socratic-debugging-benchmark                  815

✅ 완료 — 학습엔 data/hf_train.jsonl 를 사용하세요 (seed_train.jsonl 은 예시 참고용)
학습 데이터 준비 완료: data/hf_train.jsonl


## 3. 모델 로드 (4bit)


In [12]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"  # 더 작게: Qwen2.5-0.5B-Instruct / 다른 계열: Llama-3.2-1B-Instruct

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,            # 자동 감지 (T4=float16)
    load_in_4bit = True,
)


==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## 4. LoRA 어댑터 추가
전체 가중치가 아닌 일부 행렬만 학습해 메모리/시간을 절약합니다.


In [13]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


## 5. 데이터 포맷팅 (채팅 템플릿)
`instruction`(시나리오 지시) → system, `input` → user, `output` → assistant 로 매핑합니다.


In [22]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def to_text(ex):
    # 기존 ex["instruction"] 대신 무조건 시나리오[8]로 고정하여 학습시킵니다.
    fixed_instruction = (
        "시나리오[8]: 코딩 디버깅 가이드(힌트 제공형). "
        "당신은 친절한 프로그래밍 대화형 도우미입니다. "
        "정답 코드를 통째로 주기보다는, 학생이 버그의 원인을 깨닫고 스스로 해결할 수 있도록 한국어로 명확한 힌트와 원인 분석만 제공하세요."
    )

    msgs = [
        {"role": "system", "content": fixed_instruction},
        {"role": "user", "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.map(to_text)
print("샘플 수:", len(dataset))
print(dataset[0]["text"][:600])


Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

샘플 수: 1725
<|im_start|>system
시나리오[8]: 코딩 디버깅 가이드(힌트 제공형). 당신은 친절한 프로그래밍 대화형 도우미입니다. 정답 코드를 통째로 주기보다는, 학생이 버그의 원인을 깨닫고 스스로 해결할 수 있도록 한국어로 명확한 힌트와 원인 분석만 제공하세요.<|im_end|>
<|im_start|>user
Create a code in C++ to search for a user input word in an array of strings, while ignoring case sensitivity. The user input word is "apple". Additionally, the code should display the number of occurrences of the word in the array, the indices at which the word is found, and the length of each occurrence.<|im_end|>
<|im_start|>assistant
Here is the code in C++ to search for a user input word in an array of strings, while


## 6. 트레이너 설정 & 학습
데모용 `max_steps=60`. 실제 학습에선 `num_train_epochs`로 바꾸세요.


In [23]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,

        # --- [변경 구간] 데모용 max_steps는 주석처리하고 정식 에폭 학습으로 전환합니다 ---
        max_steps = 150,
        num_train_epochs = 1,

        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/1725 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,725 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,0.703536
10,0.495310
15,0.365750
20,0.325503
25,0.290900
30,0.342586
35,0.331878
40,0.312153
45,0.295089
50,0.283575


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-150/tokenizer_config.json.


## 7. 추론 테스트
학습한 시나리오 톤이 나오는지 즉석에서 확인합니다.


In [24]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "시나리오[1]: 소크라테스식 문답법으로 학생을 가이드하세요. 정답을 바로 주지 말고 힌트와 역질문으로 스스로 답을 찾게 유도합니다."},
    {"role": "user",   "content": "선생님, 삼각형 내각의 합이 왜 180도예요?"},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


system
시나리오[1]: 소크라테스식 문답법으로 학생을 가이드하세요. 정답을 바로 주지 말고 힌트와 역질문으로 스스로 답을 찾게 유도합니다.
user
선생님, 삼각형 내각의 합이 왜 180도예요?
assistant
당신이 원하는 깨달음의 길이가 아니라면, 당신이 원치 않는 방식으로 대화하지 말고 친절한 답변만 드기 바랍니다.


## 8. 저장 & 내보내기
- **LoRA 어댑터**: 가볍게 재사용 (수십 MB)
- **GGUF**: Ollama / LM Studio / llama.cpp 용 (옵션, 시간 소요)


In [25]:
# (1) LoRA 어댑터만 저장
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("LoRA 저장 완료 → lora_model/")

# (2) GGUF 내보내기 (필요할 때 주석 해제)
model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF 저장 완료 → model_gguf/")

# (3) 구글 드라이브에 백업 (필요할 때 주석 해제)
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r lora_model /content/drive/MyDrive/


LoRA 저장 완료 → lora_model/
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 6553.60it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:55<00:00, 115.27s/it]


Unsloth: Merge process complete. Saved to `/content/xideproject/xideproject/model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf_gguf/qwen2.5-1.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['model_gguf_gguf/qwen2.5-1.5b-instruct.Q4_K_M.gguf']
Unsloth: example usage f

---
### 다음 단계
1. 데이터를 더 받으려면 `--per-source` 를 키우세요 (예: 20000). 소스 추가는 `scripts/fetch_hf_datasets.py` 의 SOURCES 레지스트리에서.
2. 특정 시나리오만 학습하려면 `--only socratic math` 처럼 지정.
3. 시나리오별 성능은 `data/scenarios.json` 의 *기대치(expectation)* 기준으로 평가하세요.
4. `data/seed_train.jsonl` 은 학습용이 아니라 시나리오별 **목표 톤 예시(참고용)** 입니다.


In [28]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

# 순수한 기본 모델 로드
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)

messages = [
    {
        "role": "system",
        "content": "시나리오[8]: 코딩 디버깅 가이드(힌트 제공형)"
    },
    {
        "role": "user",
        "content": (
            "파이썬으로 리스트에서 짝수를 모두 제거하는 코드를 작성했는데, "
            "어떤 원소들은 제거되지 않고 그냥 넘어가버려요. 원인이 무엇인지 힌트를 주세요!\n\n"
            "```python\n"
            "numbers = [1, 2, 3, 4, 5, 6]\n"
            "for num in numbers:\n"
            "    if num % 2 == 0:\n"
            "        numbers.remove(num)\n"
            "```"
        )
    }
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = base_model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7)

print("=" * 50)
print("[진짜 BEFORE] 파인튜닝 전혀 안 된 기본 모델 답변:")
print("=" * 50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip())

==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[진짜 BEFORE] 파인튜닝 전혀 안 된 기본 모델 답변:
원인은 `remove()` 메서드는 이미 지정된 인덱스에만 적용됩니다. 이 메서드는 리스트의 특정 요소를 삭제하거나 변경할 수 있지만, 그렇지 않으면 예기치 못한 결과를 가져올 수 있습니다.

따라서, 모든 짝수를 제거하려면 `remove()` 메서드를 사용하여 짝수를 찾고 하나씩 제거해야 합니다. 그러나 `remove()` 메서드는 인자로 전달받은 값이 리스트 내부에 존재하는지 확인하고, 그 값이 리스트에 없을 경우 오류를 발생시키므로 `in` 연산자를 사용하여 해당 값을 찾습니다.

따라서, 수정된 코드는 다음과 같습니다:

```python
numbers = [1, 2, 3, 4, 5, 6]
for i in range(len(numbers)):
    if numbers[i] % 2 == 0:
        numbers.pop(i)  # remove the element at index i
```

위 코드에서는 `pop()` 메서드를 사용하여 인덱스에 기반한 삭제 방식을 사용합니다. 이렇게 하면 모든 짝수를 한 번에 제거할 수 있으며


In [29]:
# 내가 학습시켜 저장한 lora_model 폴더 지정
tuned_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(tuned_model)

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = tuned_model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7)

print("=" * 50)
print("[진짜 AFTER] 내가 학습시킨 모델 답변:")
print("=" * 50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip())

==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[진짜 AFTER] 내가 학습시킨 모델 답변:
The issue with the code is that it removes an element from `numbers` before iterating over it again to check for another even number. To fix this, you can use a while loop and iterate until no more elements are left in the list.

Here's one way to do it:

```python
numbers = [1, 2, 3, 4, 5, 6]

while len(numbers) > 0:
    if numbers[-1] % 2 == 0:
        numbers.pop()
```

This will remove all the even numbers from the list without skipping any.
